# 推理 Notebook（最终版）

本 Notebook 用于加载已经训练好的 `final_pipeline.joblib`、`label_encoder.joblib` 和 `training_meta.json`，对测试集执行完整推理，并导出预测结果。

设计目标：
- 与训练 Notebook 的流程尽量保持一致。
- 在推理阶段复用同样的特征工程逻辑。
- 自动对齐训练时保留的输入列，避免测试集字段不齐或多列导致报错。
- 生成可直接查看和继续加工的 `CSV` 与 `Excel` 结果文件。

## Cell 1 说明：安装依赖与导入基础库

这一段用于准备推理环境。

主要内容：
- 安装 `joblib`、`openpyxl` 等依赖，分别用于加载训练好的模型和导出 Excel。
- 导入 `pandas / numpy / json / pathlib` 等常用库。
- 屏蔽不必要的 warning，保证 Notebook 输出更整洁。

如果你的环境已经安装了这些库，可以把 `pip install` 这一行注释掉。

说明：即使未安装 `openpyxl`，本 Notebook 也会正常导出 CSV 结果，仅跳过 Excel 导出。

In [1]:
# !pip install -q joblib openpyxl pandas numpy nbformat

import os
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

warnings.filterwarnings('ignore')

## Cell 2 说明：配置路径并加载训练产物

这一段负责读取训练阶段保存的 3 个核心文件：

- `final_pipeline.joblib`：完整推理管线，内部已包含预处理器和最终模型。
- `label_encoder.joblib`：把整数类别还原为字符串标签。
- `training_meta.json`：记录训练时删掉了哪些列、最终保留了哪些特征列。

请根据你的实际目录修改下面的路径。

In [2]:
MODEL_DIR = Path('output/models_023')
TEST_PATH = Path('output/features_test1000_v2.csv')         # 已修改为测试集实际路径
OUTPUT_DIR = Path('output/infer_results')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PIPELINE_PATH = MODEL_DIR / 'final_pipeline.joblib'
LABEL_ENCODER_PATH = MODEL_DIR / 'label_encoder.joblib'
META_PATH = MODEL_DIR / 'training_meta.json'

final_pipeline = joblib.load(PIPELINE_PATH)
label_encoder = joblib.load(LABEL_ENCODER_PATH)
with open(META_PATH, 'r', encoding='utf-8') as f:
    training_meta = json.load(f)

print('Loaded pipeline from:', PIPELINE_PATH)
print('Loaded label encoder from:', LABEL_ENCODER_PATH)
print('Loaded meta from:', META_PATH)
print('Selected model:', training_meta.get('selected_model'))
print('Number of training feature columns:', len(training_meta.get('feature_columns_before_preprocess', [])))

Loaded pipeline from: output\models_023\final_pipeline.joblib
Loaded label encoder from: output\models_023\label_encoder.joblib
Loaded meta from: output\models_023\training_meta.json
Selected model: XGBoost
Number of training feature columns: 154


## Cell 3 说明：定义与训练阶段一致的特征工程函数

这一段非常关键。  
因为训练 Notebook 在模型训练前，对原始字段做过一轮“显式特征工程”，而这些派生特征并不在 `Pipeline` 内部，所以推理阶段必须手动再做一次。

这里保留与训练版一致的核心逻辑：
- 窗口比值特征
- 伤害净值特征
- 技能与行为密度特征
- 路径效率特征
- 趋势差分特征
- 规则型二值标记

注意：训练阶段还对类别列做了稀有值压缩；推理时由于没有保存每列的保留类别词表，这里不再强行压缩，而是依赖训练管线中的 `OneHotEncoder(handle_unknown="ignore")` 自动忽略未见类别。

In [3]:
def safe_div(a, b):
    if isinstance(b, pd.Series):
        return a / b.replace(0, np.nan)
    return a / (np.nan if b == 0 else b)


def add_engineered_features(df_in: pd.DataFrame) -> pd.DataFrame:
    df = df_in.copy()

    if {'recent3speedmean', 'recent20speedmean'}.issubset(df.columns):
        df['speed_ratio_3_20'] = safe_div(df['recent3speedmean'], df['recent20speedmean'] + 1e-6)
    if {'recent5speedmean', 'recent20speedmean'}.issubset(df.columns):
        df['speed_ratio_5_20'] = safe_div(df['recent5speedmean'], df['recent20speedmean'] + 1e-6)

    if {'recent3scoperatio', 'recent20scoperatio'}.issubset(df.columns):
        df['scope_ratio_3_20'] = safe_div(df['recent3scoperatio'] + 1e-6, df['recent20scoperatio'] + 1e-6)
    if {'recent5scoperatio', 'recent20scoperatio'}.issubset(df.columns):
        df['scope_ratio_5_20'] = safe_div(df['recent5scoperatio'] + 1e-6, df['recent20scoperatio'] + 1e-6)

    if {'recent3fovmean', 'recent20fovmean'}.issubset(df.columns):
        df['fov_ratio_3_20'] = safe_div(df['recent3fovmean'], df['recent20fovmean'] + 1e-6)
    if {'recent5fovmean', 'recent20fovmean'}.issubset(df.columns):
        df['fov_ratio_5_20'] = safe_div(df['recent5fovmean'], df['recent20fovmean'] + 1e-6)

    if {'recent3damageoutcount', 'recent3damageincount'}.issubset(df.columns):
        df['recent3_damage_count_net'] = df['recent3damageoutcount'] - df['recent3damageincount']
    if {'recent5damageoutcount', 'recent5damageincount'}.issubset(df.columns):
        df['recent5_damage_count_net'] = df['recent5damageoutcount'] - df['recent5damageincount']
    if {'recent20damageouthpsum', 'recent20damageinhpsum'}.issubset(df.columns):
        df['recent20_damage_hp_net_v2'] = df['recent20damageouthpsum'] - df['recent20damageinhpsum']
    if {'recent5damageouthpsum', 'recent5damageinhpsum'}.issubset(df.columns):
        df['recent5_damage_hp_net_v2'] = df['recent5damageouthpsum'] - df['recent5damageinhpsum']

    if {'recent3damageoutcount', 'recent20damageoutcount'}.issubset(df.columns):
        df['damage_out_ratio_3_20'] = safe_div(df['recent3damageoutcount'] + 1e-6, df['recent20damageoutcount'] + 1e-6)
    if {'recent3damageincount', 'recent20damageincount'}.issubset(df.columns):
        df['damage_in_ratio_3_20'] = safe_div(df['recent3damageincount'] + 1e-6, df['recent20damageincount'] + 1e-6)

    if {'recent5skillcastcount', 'recent20skillcastcount'}.issubset(df.columns):
        df['skill_cast_ratio_5_20'] = safe_div(df['recent5skillcastcount'] + 1e-6, df['recent20skillcastcount'] + 1e-6)
    if {'recent5actioncount', 'recent20frames'}.issubset(df.columns):
        df['action_density_5'] = safe_div(df['recent5actioncount'], df['recent20frames'] + 1e-6)
    if {'recent3actioncount', 'recent20frames'}.issubset(df.columns):
        df['action_density_3'] = safe_div(df['recent3actioncount'], df['recent20frames'] + 1e-6)

    if {'recent20pathlen', 'recent20dispnorm'}.issubset(df.columns):
        df['path_efficiency_20'] = safe_div(df['recent20dispnorm'] + 1e-6, df['recent20pathlen'] + 1e-6)
    if {'recent5pathlen', 'recent5dispnorm'}.issubset(df.columns):
        df['path_efficiency_5'] = safe_div(df['recent5dispnorm'] + 1e-6, df['recent5pathlen'] + 1e-6)
    if {'nearestplayerdist', 'meanplayerdist'}.issubset(df.columns):
        df['nearest_mean_dist_ratio'] = safe_div(df['nearestplayerdist'] + 1e-6, df['meanplayerdist'] + 1e-6)

    if {'seg1speedmean', 'seg2speedmean'}.issubset(df.columns):
        df['seg_speed_mean_diff'] = df['seg2speedmean'] - df['seg1speedmean']
        df['seg_speed_mean_absdiff'] = (df['seg2speedmean'] - df['seg1speedmean']).abs()
    if {'seg1scoperatio', 'seg2scoperatio'}.issubset(df.columns):
        df['seg_scope_ratio_diff'] = df['seg2scoperatio'] - df['seg1scoperatio']
        df['seg_scope_ratio_absdiff'] = (df['seg2scoperatio'] - df['seg1scoperatio']).abs()
    if {'seg1fovmean', 'seg2fovmean'}.issubset(df.columns):
        df['seg_fov_mean_diff'] = df['seg2fovmean'] - df['seg1fovmean']
        df['seg_fov_mean_absdiff'] = (df['seg2fovmean'] - df['seg1fovmean']).abs()

    if 'recent3damageincount' in df.columns:
        df['flag_recent3_under_attack'] = (df['recent3damageincount'] > 0).astype(int)
    if 'recent3damageoutcount' in df.columns:
        df['flag_recent3_attack_out'] = (df['recent3damageoutcount'] > 0).astype(int)
    if 'recent3scoperatio' in df.columns:
        df['flag_recent3_scoped'] = (df['recent3scoperatio'] > 0).astype(int)
    if 'recent20skillcastcount' in df.columns:
        df['flag_recent20_skill_used'] = (df['recent20skillcastcount'] > 0).astype(int)

    return df

## Cell 4 说明：读取测试集，并保留标识列

这一段用于加载测试集。  
同时会尽量保留 `sample_id`、`main_player_id` 等标识信息，方便后面导出结果时做关联。

如果测试集里没有 `label` 列，这是正常情况；Notebook 会自动按“纯推理模式”执行。

In [4]:
test_df_raw = pd.read_csv(TEST_PATH)
print('test shape =', test_df_raw.shape)
display(test_df_raw.head())

possible_id_cols = [c for c in ['sample_id', 'main_player_id', 'decision_time'] if c in test_df_raw.columns]
id_df = test_df_raw[possible_id_cols].copy() if possible_id_cols else pd.DataFrame(index=test_df_raw.index)

print('kept id columns =', possible_id_cols)

test shape = (1000, 139)


,sample_id,label,main_player_id,decision_time,last_x,last_y,last_z,last_weapon_yaw,last_weapon_pitch,last_vx,...,last_death_in_gap,nearest_player_dist,mean_player_dist,std_player_dist,recent5_path_len,recent20_path_len,recent5_disp_norm,recent20_disp_norm,path_straightness_20,decision_effect_radius
0,1,测试1000题,4587.0,20.0,3527.7,-212.9,-6401.6,88.5,1.0,0.0,...,NaN,1.920937,314.222066,288.004812,85.229944,818.768180,0.843861,16.258844,1.985769e-02,NaN
1,2,测试1000题,7335.0,20.0,3655.5,-250.5,-4722.5,117.0,8.3,0.0,...,NaN,25.958814,338.791788,148.797508,94.056914,692.849790,0.931257,9.024411,1.302506e-02,NaN
2,3,测试1000题,274.0,20.0,6741.2,-213.3,-4406.8,130.9,350.0,-4.6,...,NaN,31.684381,63.659159,31.974778,577.698314,2051.516607,5.719785,73.510407,3.583223e-02,NaN
3,4,测试1000题,1790.0,20.0,3647.2,-161.1,-7897.4,119.8,358.3,-0.4,...,NaN,16.876018,219.051563,240.859624,393.112877,1148.808033,3.892207,13.952061,1.214481e-02,NaN
4,5,测试1000题,6871.0,20.0,3840.5,-254.8,-4711.0,77.9,0.0,0.0,...,NaN,4.404543,212.781634,120.683036,0.000000,0.000000,0.000000,6083.410580,6.083411e+09,NaN


kept id columns = ['sample_id', 'main_player_id', 'decision_time']


## Cell 5 说明：按照训练阶段逻辑构造推理输入

这里会做三件事：

1. 删除训练时不参与建模的高风险字段。
2. 在剩余字段上执行与训练一致的特征工程。
3. 用 `training_meta.json` 里保存的 `feature_columns_before_preprocess` 对齐列集合。

这一步非常重要，因为模型在训练时见过的特征列顺序、列名和推理时必须一致。  
缺失的列会自动补成 `NaN`，多出来的列会自动删掉。

In [5]:
TARGET_COL = training_meta.get('target_col', 'label')
initial_drop_cols = training_meta.get('initial_drop_cols', training_meta.get('drop_cols', []))
required_feature_cols = training_meta.get('feature_columns_before_preprocess', [])

work_df = test_df_raw.copy()
if TARGET_COL in work_df.columns:
    work_df = work_df.drop(columns=[TARGET_COL])

feature_base_cols = [c for c in work_df.columns if c not in initial_drop_cols]
feature_base_df = work_df[feature_base_cols].copy()
feature_fe_df = add_engineered_features(feature_base_df)

missing_required_cols = [c for c in required_feature_cols if c not in feature_fe_df.columns]
for c in missing_required_cols:
    feature_fe_df[c] = np.nan

X_infer = feature_fe_df[required_feature_cols].copy()

print('aligned inference shape =', X_infer.shape)
print('all required columns matched =', len(missing_required_cols) == 0)
print('missing required feature columns =', len(missing_required_cols), '/', len(required_feature_cols))

# 缺失过多通常意味着用了不同版本的特征文件，评估结果会明显失真
missing_ratio = len(missing_required_cols) / max(1, len(required_feature_cols))
if missing_ratio > 0.2:
    print('WARNING: feature schema mismatch is large. Please use *_v2 feature file for this model.')

aligned inference shape = (1000, 154)
all required columns matched = False
missing required feature columns = 24 / 154


## Cell 6 说明：执行预测，并输出类别概率

这一段会调用训练好的完整 pipeline 进行推理。  
输出包括：
- `pred_label`：最终预测标签。
- `pred_confidence`：最高类别概率。
- `top1 / top2 / top3`：前三个最可能类别，便于后续人工复核和规则后处理。

In [6]:
pred_idx = final_pipeline.predict(X_infer)
pred_label = label_encoder.inverse_transform(pred_idx)

proba = final_pipeline.predict_proba(X_infer)
class_names = list(label_encoder.classes_)

proba_df = pd.DataFrame(proba, columns=[f'proba_{c}' for c in class_names])
rank_idx = np.argsort(-proba, axis=1)

results_df = id_df.copy()
results_df['pred_label'] = pred_label
results_df['pred_confidence'] = proba.max(axis=1)
results_df['top1_label'] = [class_names[i[0]] for i in rank_idx]
results_df['top1_score'] = [proba[r, i[0]] for r, i in enumerate(rank_idx)]
results_df['top2_label'] = [class_names[i[1]] if len(i) > 1 else None for i in rank_idx]
results_df['top2_score'] = [proba[r, i[1]] if len(i) > 1 else None for r, i in enumerate(rank_idx)]
results_df['top3_label'] = [class_names[i[2]] if len(i) > 2 else None for i in rank_idx]
results_df['top3_score'] = [proba[r, i[2]] if len(i) > 2 else None for r, i in enumerate(rank_idx)]

results_full_df = pd.concat([results_df, proba_df], axis=1)
print('prediction done. preview:')
display(results_full_df.head())

prediction done. preview:


,sample_id,main_player_id,decision_time,pred_label,pred_confidence,top1_label,top1_score,top2_label,top2_score,top3_label,top3_score,proba_Action,proba_BeingResuce,proba_Fire,proba_Grenade,proba_Looting,proba_SkillStart
0,1,4587.0,20.0,Action,0.846671,Action,0.846671,Fire,0.119272,SkillStart,0.033123,0.846671,0.000025,0.119272,0.000198,0.000712,0.033123
1,2,7335.0,20.0,SkillStart,0.458866,SkillStart,0.458866,Fire,0.361209,Action,0.138372,0.138372,0.000261,0.361209,0.038053,0.003239,0.458866
2,3,274.0,20.0,SkillStart,0.959926,SkillStart,0.959926,Fire,0.039338,Grenade,0.000456,0.000194,0.000005,0.039338,0.000456,0.000082,0.959926
3,4,1790.0,20.0,Action,0.594325,Action,0.594325,Fire,0.405142,Grenade,0.000456,0.594325,0.000003,0.405142,0.000456,0.000059,0.000015
4,5,6871.0,20.0,Looting,0.995615,Looting,0.995615,SkillStart,0.002022,Grenade,0.001792,0.000063,0.000140,0.000369,0.001792,0.995615,0.002022


## Cell 7 说明：生成通用结果与比赛提交文件

这一段会输出三类结果：
- `predictions_full.csv / .xlsx`：包含概率、top3 结果，适合分析。
- `submission_basic.csv / .xlsx`：保留标识列和最终预测标签。
- `submission_competition.csv / .xlsx`：比赛格式结果，列名为：`题目序号`、`意图决策`、`动作行为`。

其中 `意图决策` 的映射规则：
- `Looting`、`BeingResuce` -> `避战`
- 其他行为 -> `交战`

说明：若环境未安装 `openpyxl`，会自动跳过 Excel 导出，仅保留 CSV。

In [7]:
predictions_full_csv = OUTPUT_DIR / 'predictions_full.csv'
predictions_full_xlsx = OUTPUT_DIR / 'predictions_full.xlsx'
submission_basic_csv = OUTPUT_DIR / 'submission_basic.csv'
submission_basic_xlsx = OUTPUT_DIR / 'submission_basic.xlsx'
submission_comp_csv = OUTPUT_DIR / 'submission_competition.csv'
submission_comp_xlsx = OUTPUT_DIR / 'submission_competition.xlsx'

results_full_df.to_csv(predictions_full_csv, index=False, encoding='utf-8-sig')

submission_cols = [c for c in ['sample_id', 'main_player_id', 'decision_time'] if c in results_df.columns] + ['pred_label']
submission_basic_df = results_df[submission_cols].copy()
submission_basic_df.to_csv(submission_basic_csv, index=False, encoding='utf-8-sig')

qid = results_df['sample_id'].astype(str) if 'sample_id' in results_df.columns else pd.Series(np.arange(1, len(results_df) + 1), dtype='int64').astype(str)
qid = qid.where(~qid.str.fullmatch(r'\d+'), qid.str.zfill(4))

combat_labels = {'Looting', 'BeingResuce'}
submission_comp_df = pd.DataFrame({
    '题目序号': qid,
    '意图决策': results_df['pred_label'].astype(str).map(lambda x: '避战' if x in combat_labels else '交战'),
    '动作行为': results_df['pred_label'].astype(str)
})
submission_comp_df.to_csv(submission_comp_csv, index=False, encoding='utf-8-sig')

excel_ok = True
try:
    import openpyxl  # noqa: F401
except ModuleNotFoundError:
    excel_ok = False

if excel_ok:
    results_full_df.to_excel(predictions_full_xlsx, index=False)
    submission_basic_df.to_excel(submission_basic_xlsx, index=False)
    submission_comp_df.to_excel(submission_comp_xlsx, index=False)

print('saved files:')
print('-', predictions_full_csv.resolve())
print('-', submission_basic_csv.resolve())
print('-', submission_comp_csv.resolve())
if excel_ok:
    print('-', predictions_full_xlsx.resolve())
    print('-', submission_basic_xlsx.resolve())
    print('-', submission_comp_xlsx.resolve())
else:
    print('- Excel export skipped (openpyxl is not installed).')

print('\nsubmission_competition preview:')
display(submission_comp_df.head())

saved files:
- E:\EOne\2026游戏安全技术竞赛-游戏安全AI方向-初赛\output\infer_results\predictions_full.csv
- E:\EOne\2026游戏安全技术竞赛-游戏安全AI方向-初赛\output\infer_results\submission_basic.csv
- E:\EOne\2026游戏安全技术竞赛-游戏安全AI方向-初赛\output\infer_results\submission_competition.csv
- E:\EOne\2026游戏安全技术竞赛-游戏安全AI方向-初赛\output\infer_results\predictions_full.xlsx
- E:\EOne\2026游戏安全技术竞赛-游戏安全AI方向-初赛\output\infer_results\submission_basic.xlsx
- E:\EOne\2026游戏安全技术竞赛-游戏安全AI方向-初赛\output\infer_results\submission_competition.xlsx

submission_competition preview:


,题目序号,意图决策,动作行为
0,0001,交战,Action
1,0002,交战,SkillStart
2,0003,交战,SkillStart
3,0004,交战,Action
4,0005,避战,Looting


## Cell 8 说明：可选——如果测试集带真值标签，可直接本地评估

有时你会拿一份验证集来模拟真实推理，这时文件中可能仍然带有 `label` 列。  
下面这段代码会在存在真值标签时自动计算：
- accuracy
- macro_f1
- 分类报告

如果测试集没有真值，这一段会自动跳过。

In [8]:
if TARGET_COL in test_df_raw.columns:
    y_true_raw = test_df_raw[TARGET_COL].astype(str)
    y_pred_raw = pd.Series(pred_label).astype(str)

    train_classes = set(map(str, list(label_encoder.classes_)))
    true_classes = set(map(str, y_true_raw.unique().tolist()))

    # 只要测试集中存在任一训练类标签，就执行评估；其余未知标签样本先过滤
    valid_mask = y_true_raw.isin(train_classes)
    valid_count = int(valid_mask.sum())
    total_count = int(len(y_true_raw))

    if valid_count == 0:
        print('Skip evaluation: no overlap between true labels and train classes.')
        print('true classes  =', sorted(true_classes))
        print('train classes =', sorted(train_classes))
    else:
        if valid_count < total_count:
            unknown_labels = sorted(set(y_true_raw[~valid_mask].unique().tolist()))
            print(f'Warning: filtered out {total_count - valid_count} rows with unknown labels: {unknown_labels}')
            print(f'Evaluating on overlapped rows only: {valid_count}/{total_count}')

        from sklearn.metrics import accuracy_score, f1_score, classification_report
        y_true_eval = y_true_raw[valid_mask]
        y_pred_eval = y_pred_raw[valid_mask]

        print('Validation accuracy =', accuracy_score(y_true_eval, y_pred_eval))
        print('Validation macro_f1 =', f1_score(y_true_eval, y_pred_eval, average='macro'))
        print('\nClassification report:')
        print(classification_report(y_true_eval, y_pred_eval, digits=4))

        # 业务重点：评估“交战/避战”二分类效果
        avoid_labels = {'BeingResuce', 'Looting'}
        y_true_intent = y_true_eval.map(lambda x: '避战' if x in avoid_labels else '交战')
        y_pred_intent = y_pred_eval.map(lambda x: '避战' if x in avoid_labels else '交战')

        print('\nIntent-level metrics (交战/避战):')
        print('Intent accuracy =', accuracy_score(y_true_intent, y_pred_intent))
        print('Intent macro_f1 =', f1_score(y_true_intent, y_pred_intent, average='macro'))
        print('Intent classification report:')
        print(classification_report(y_true_intent, y_pred_intent, digits=4))
else:
    print('No ground-truth label found in test file. Skip evaluation.')

Skip evaluation: no overlap between true labels and train classes.
true classes  = ['测试1000题']
train classes = ['Action', 'BeingResuce', 'Fire', 'Grenade', 'Looting', 'SkillStart']


## Cell 9 说明：可选——映射为比赛专用提交格式

如果你的比赛要求某个固定字段名，例如：
- `sample_id`
- `player_id`
- `decision_type`
- `behavior_label`

那么可以在这一段基于 `submission_basic_df` 继续改写。  
当前 Notebook 先给出一个通用版本，避免误写成与你手头模板不一致的格式。

In [9]:
# 对 Cell7 导出的比赛文件做二次格式化：仅修改“动作行为”列，并导出 csv/xlsx
submission_comp_csv = OUTPUT_DIR / 'submission_competition.csv'
submission_comp_xlsx = OUTPUT_DIR / 'submission_competition.xlsx'

submit_df = pd.read_csv(submission_comp_csv, encoding='utf-8-sig')

action_map = {
    '开镜': 'Action',
    '开火': 'Fire',
    '使用技能': 'SkilIStart',
    '丢雷': 'Grenade',
    '搜物资': 'Looting',
    '救援': 'BeingResuce',
}

if '动作行为' not in submit_df.columns:
    raise KeyError('submission_competition.csv 中未找到列: 动作行为')

submit_df['动作行为'] = submit_df['动作行为'].astype(str).str.strip().map(
    lambda x: action_map.get(x, x)
)

# submit_df.to_csv(submission_comp_csv, index=False, encoding='utf-8-sig')
# print('已保存 CSV:', submission_comp_csv.resolve())

try:
    import openpyxl  # noqa: F401
    submit_df.to_excel(submission_comp_xlsx, index=False)
    print('已保存 XLSX:', submission_comp_xlsx.resolve())
except ModuleNotFoundError:
    print('未安装 openpyxl，跳过 XLSX 导出。可先执行: pip install openpyxl')

print('已完成 submission_competition 格式化，仅修改列: 动作行为')
display(submit_df.head())

已保存 XLSX: E:\EOne\2026游戏安全技术竞赛-游戏安全AI方向-初赛\output\infer_results\submission_competition.xlsx
已完成 submission_competition 格式化，仅修改列: 动作行为


,题目序号,意图决策,动作行为
0,1,交战,Action
1,2,交战,SkillStart
2,3,交战,SkillStart
3,4,交战,Action
4,5,避战,Looting
